# Road Accident Severity Prediction — Final Preprocessing

This notebook follows the preprocessing sequence demonstrated in the class preprocessing notebook:

1. remove duplicates
2. remove the extremely small non-injury target class
3. separate `X` and `y`
4. train/test split before preprocessing
5. remove identifier and leakage columns
6. apply date/time feature engineering
7. use the features selected by Information Gain
8. median imputation for numerical features
9. most-frequent imputation for categorical features
10. one-hot encoding for nominal features
11. label encoding for the target
12. standard scaling for numerical features
13. save the four processed train/test files

All learned preprocessing parameters are fitted on the training data only.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

DATA_PATH = "../data/victorian_road_crash_data.csv"
SELECTED_FEATURES_PATH = "../data/processed/selected_features.csv"

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)
display(df.head())


Original dataset shape: (200352, 52)


,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,DAY_OF_WEEK,DCA_CODE,DCA_CODE_DESCRIPTION,LIGHT_CONDITION,POLICE_ATTEND,ROAD_GEOMETRY,...,NO_OF_VEHICLES,HEAVYVEHICLE,PASSENGERVEHICLE,MOTORCYCLE,PT_VEHICLE,DEG_URBAN_NAME,SRNS,RMA,DIVIDED,STAT_DIV_NAME
0,T20140024624,27-11-2014,18:35:00,Collision with vehicle,Thursday,110,CROSS TRAFFIC(INTERSECTIONS ONLY),Day,Yes,Cross intersection,...,2.0,0.0,2.0,0.0,0.0,TOWNS,NaN,Local Road,Undivided,Country
1,T20190026336,27-12-2019,15:45:00,Collision with vehicle,Friday,113,RIGHT NEAR (INTERSECTIONS ONLY),Day,Yes,T intersection,...,2.0,0.0,2.0,0.0,0.0,MELB_URBAN,NaN,Arterial Other,Divided,Metro
2,T20190019196,02-10-2019,12:07:00,Collision with vehicle,Wednesday,173,RIGHT OFF CARRIAGEWAY INTO OBJECT/PARKED VEHICLE,Day,Yes,Not at intersection,...,3.0,0.0,2.0,0.0,0.0,MELB_URBAN,M,Freeway,Divided,Metro
3,T20250029202,07-11-2025,13:10:00,Struck Pedestrian,Friday,100,PED NEAR SIDE. PED HIT BY VEHICLE FROM THE RIGHT.,Day,Yes,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,MELB_URBAN,NaN,Arterial Highway,Divided,Metro
4,T20210005363,30-01-2021,06:30:00,Collision with a fixed object,Saturday,183,OFF LEFT BEND INTO OBJECT/PARKED VEHICLE,Dusk/Dawn,No,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,RURAL_VICTORIA,C,Arterial Other,Undivided,Metro


In [2]:
# Remove exact duplicates

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)


Duplicate rows: 0
Shape after removing duplicates: (200352, 52)


In [3]:
# Remove the very small non-injury target class

print("Target distribution before removal:")
print(df["SEVERITY"].value_counts())

df = df[df["SEVERITY"] != "Non injury accident"].copy()

print("\nTarget distribution after removal:")
print(df["SEVERITY"].value_counts())
print("\nDataset shape:", df.shape)


Target distribution before removal:
SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Non injury accident             4
Name: count, dtype: int64

Target distribution after removal:
SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Name: count, dtype: int64

Dataset shape: (200348, 52)


In [4]:
# Separate features and target

X = df.drop(columns=["SEVERITY"])
y = df["SEVERITY"]

print("X:", X.shape)
print("y:", y.shape)


X: (200348, 51)
y: (200348,)


In [5]:
# Train/test split BEFORE imputation, encoding and scaling

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (160278, 51)
X_test : (40070, 51)
y_train: (160278,)
y_test : (40070,)


In [6]:
# Remove identifier and target-leakage columns

drop_columns = [
    "ACCIDENT_NO",
    "INJ_OR_FATAL",
    "FATALITY",
    "SERIOUSINJURY",
    "OTHERINJURY",
    "NONINJURED"
]

X_train = X_train.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

print("Removed columns:", drop_columns)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Removed columns: ['ACCIDENT_NO', 'INJ_OR_FATAL', 'FATALITY', 'SERIOUSINJURY', 'OTHERINJURY', 'NONINJURED']
X_train: (160278, 45)
X_test : (40070, 45)


## Date/time feature engineering

Following the existing road-accident preprocessing workflow, date and time are converted into useful numerical features:

- accident year
- accident month
- accident day
- accident hour
- accident minute
- weekend indicator

The original date/time columns are then removed.


In [7]:
def add_datetime_features(data):
    data = data.copy()

    data["ACCIDENT_DATE"] = pd.to_datetime(
        data["ACCIDENT_DATE"],
        errors="coerce"
    )

    data["ACCIDENT_YEAR"] = data["ACCIDENT_DATE"].dt.year
    data["ACCIDENT_MONTH"] = data["ACCIDENT_DATE"].dt.month
    data["ACCIDENT_DAY"] = data["ACCIDENT_DATE"].dt.day

    data["ACCIDENT_TIME"] = pd.to_datetime(
        data["ACCIDENT_TIME"],
        format="mixed",
        errors="coerce"
    )

    data["ACCIDENT_HOUR"] = data["ACCIDENT_TIME"].dt.hour
    data["ACCIDENT_MINUTE"] = data["ACCIDENT_TIME"].dt.minute

    data["IS_WEEKEND"] = data["DAY_OF_WEEK"].isin(
        ["Saturday", "Sunday"]
    ).astype(int)

    data = data.drop(
        columns=["ACCIDENT_DATE", "ACCIDENT_TIME"],
        errors="ignore"
    )

    return data


X_train = add_datetime_features(X_train)
X_test = add_datetime_features(X_test)

print("After date/time feature engineering:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


After date/time feature engineering:
X_train: (160278, 49)
X_test : (40070, 49)


## Apply Information Gain feature selection

The selected feature list is produced by the feature-selection notebook.

We keep the selection at the **original feature level**, before one-hot encoding, so the feature-selection result remains interpretable.


In [8]:
selected_features = pd.read_csv(
    SELECTED_FEATURES_PATH
)["Feature"].tolist()

# Keep only selected features that exist after feature engineering
selected_features = [
    col for col in selected_features
    if col in X_train.columns
]

X_train = X_train[selected_features].copy()
X_test = X_test[selected_features].copy()

print("Selected feature count:", len(selected_features))
print("Selected features:")
for feature in selected_features:
    print("-", feature)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)


Selected feature count: 20
Selected features:
- POLICE_ATTEND
- DCA_CODE_DESCRIPTION
- DCA_CODE
- STAT_DIV_NAME
- NO_OF_VEHICLES
- DRIVER
- ACCIDENT_TYPE
- DEG_URBAN_NAME
- DIVIDED
- PASSENGERVEHICLE
- ROAD_NAME
- LIGHT_CONDITION
- SPEED_ZONE
- LONGITUDE
- VICGRID_Y
- LATITUDE
- VICGRID_X
- ROAD_ROUTE_1
- SRNS
- MALES

X_train: (160278, 20)
X_test : (40070, 20)


In [9]:
# Identify numerical and categorical columns

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)


Numerical features: 10
['DCA_CODE', 'NO_OF_VEHICLES', 'DRIVER', 'PASSENGERVEHICLE', 'LONGITUDE', 'VICGRID_Y', 'LATITUDE', 'VICGRID_X', 'ROAD_ROUTE_1', 'MALES']

Categorical features: 10
['POLICE_ATTEND', 'DCA_CODE_DESCRIPTION', 'STAT_DIV_NAME', 'ACCIDENT_TYPE', 'DEG_URBAN_NAME', 'DIVIDED', 'ROAD_NAME', 'LIGHT_CONDITION', 'SPEED_ZONE', 'SRNS']


## Missing-value audit

The audit is performed on the training data. The preprocessing itself is fitted only on the training data and then applied unchanged to the test data.


In [10]:
missing_data = pd.DataFrame({
    "Missing Values": X_train.isnull().sum(),
    "Percentage": (
        X_train.isnull().sum() / len(X_train) * 100
    ).round(2)
})

missing_data = missing_data[
    missing_data["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

display(missing_data)


,Missing Values,Percentage
SRNS,112226,70.02
DIVIDED,6152,3.84
STAT_DIV_NAME,803,0.50
DEG_URBAN_NAME,786,0.49
ROAD_NAME,214,0.13
VICGRID_X,71,0.04
VICGRID_Y,71,0.04
LONGITUDE,71,0.04
ROAD_ROUTE_1,71,0.04
LATITUDE,71,0.04


## Preprocessing pipelines

This follows the class workflow:

- numerical: median imputation → standard scaling
- categorical: most-frequent imputation → one-hot encoding
- `handle_unknown="ignore"` ensures unseen test categories do not break preprocessing
- `drop="first"` follows the class notebook's dummy-variable handling


In [11]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)


In [12]:
# Fit ONLY on training data

X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same learned preprocessing to test data
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_test :", X_test_processed.shape)


c:\Users\Rehan's Lenovo\OneDrive\Desktop\KLH\2-1\ML\github\Road-Accident-Severity-Prediction\.MLvenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Processed X_train: (160278, 13386)
Processed X_test : (40070, 13386)


In [13]:
# Label encode the target using the training labels

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Target mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

print("\ny_train:", y_train_encoded.shape)
print("y_test :", y_test_encoded.shape)


Target mapping:
0 -> Fatal accident
1 -> Other injury accident
2 -> Serious injury accident

y_train: (160278,)
y_test : (40070,)


In [14]:
# Final validation checks

print("Final processed shapes")
print("=" * 40)
print("X_train:", X_train_processed.shape)
print("X_test :", X_test_processed.shape)
print("y_train:", y_train_encoded.shape)
print("y_test :", y_test_encoded.shape)

print("\nNaN values in X_train:",
      np.isnan(X_train_processed).sum())

print("NaN values in X_test:",
      np.isnan(X_test_processed).sum())


Final processed shapes
X_train: (160278, 13386)
X_test : (40070, 13386)
y_train: (160278,)
y_test : (40070,)

NaN values in X_train: 0
NaN values in X_test: 0


In [15]:
# Get the final feature names

feature_names = preprocessor.get_feature_names_out()

print("Number of final model features:", len(feature_names))
print("\nFirst 20 feature names:")
print(feature_names[:20])


Number of final model features: 13386

First 20 feature names:
['num__DCA_CODE' 'num__NO_OF_VEHICLES' 'num__DRIVER'
 'num__PASSENGERVEHICLE' 'num__LONGITUDE' 'num__VICGRID_Y' 'num__LATITUDE'
 'num__VICGRID_X' 'num__ROAD_ROUTE_1' 'num__MALES'
 'cat__POLICE_ATTEND_Not known' 'cat__POLICE_ATTEND_Yes'
 'cat__DCA_CODE_DESCRIPTION_ANY MANOEUVRE INVOLVING PED NOT INCLUDED IN DCAs 100-108.'
 'cat__DCA_CODE_DESCRIPTION_CROSS TRAFFIC(INTERSECTIONS ONLY)'
 'cat__DCA_CODE_DESCRIPTION_CUTTING IN (OVERTAKING)'
 'cat__DCA_CODE_DESCRIPTION_DOUBLE PARKED'
 'cat__DCA_CODE_DESCRIPTION_ENTERING PARKING'
 'cat__DCA_CODE_DESCRIPTION_FAR SIDE. PED HIT BY VEHICLE FROM THE LEFT'
 'cat__DCA_CODE_DESCRIPTION_FELL IN/FROM VEHICLE'
 'cat__DCA_CODE_DESCRIPTION_HEAD ON (NOT OVERTAKING)']


## Save processed data

The files are saved in the same four-file format used by the existing project workflow:

- `X_train.csv`
- `X_test.csv`
- `y_train.csv`
- `y_test.csv`

The preprocessor is also saved so the exact same transformations can be reused later.


In [ ]:
# Save X

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

X_train_processed_df.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

# Save y

pd.DataFrame(
    y_train_encoded,
    columns=["SEVERITY"]
).to_csv(
    "../data/processed/y_train.csv",
    index=False
)

pd.DataFrame(
    y_test_encoded,
    columns=["SEVERITY"]
).to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("All four processed files saved successfully.")


In [ ]:
import joblib

joblib.dump(
    preprocessor,
    "../data/processed/preprocessor.joblib"
)

joblib.dump(
    label_encoder,
    "../data/processed/label_encoder.joblib"
)

print("Saved preprocessing objects successfully.")


## Final workflow

```text
RAW DATA
   ↓
Remove duplicates
   ↓
Remove "Non injury accident"
   ↓
Separate X / y
   ↓
Train / Test Split
   ↓
Remove ACCIDENT_NO + leakage columns
   ↓
Date / time feature engineering
   ↓
Information Gain selected features
   ↓
 ┌─────────────────────────────┐
 │ Numerical: median → scale   │
 │ Categorical: mode → one-hot │
 └─────────────────────────────┘
   ↓
Encode target
   ↓
Save X_train / X_test / y_train / y_test
```
